# 🧠 [SmartRouter Pipeline: Step 1] 고객 이탈 변수 중요도 분석

기존 교정 서비스에서 발생하는 고객 이탈(Churn) 현상의 원인을 데이터 기반으로 분석합니다.
**가설**: 교정 난이도가 높은(어려운) 문장을 경량 모델(Gemini Flash)로 일괄 처리하면서 발생하는 품질 저하가 이탈의 주 원인일 것이다.

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score

import lightgbm
import catboost
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 2025
np.random.seed(SEED)
plt.rc('font', family='NanumGothic')
plt.style.use('ggplot')
print("--- 라이브러리 로드 완료 ---")

In [ ]:
#  데이터 로드 및 "타겟 변수(y)" 생성
print("--- [1/9] 데이터 로드 및 '타겟 변수(y)' 생성 ---")
file_path = './data/df_analysis.parquet'
try:
    df_analysis = pd.read_parquet(file_path)
    print(f"  - 'df_analysis' (Shape: {df_analysis.shape}) 로드 완료!")
except Exception as e:
    print(f"  - ⚠️ 오류: {e}. 02번 파일을 먼저 실행하세요.")

# [핵심!] "AI 제안 수락률"을 0과 1로 만듭니다.
# (y = 0: 'run' 이벤트, y = 1: 'selected' 이벤트)
# ('event_name'을 'y'로 변환하고, X에서는 'event_name'을 제거하여 "컨닝"을 막습니다)
df_analysis['is_selected'] = np.where(df_analysis['event_name'].str.startswith('run_'), 0, 1)

print(f"  - 'y' (is_selected) 생성 완료. (실행: 0, 선택: 1)")
display(df_analysis['is_selected'].value_counts(normalize=True))

In [ ]:
#  훈련/테스트 분리 및 모델 정의
print("\n--- [4/9] 훈련/테스트 분리 및 모델 정의 ---")

# 1. 훈련용(80%)과 테스트용(20%) 데이터로 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"  - 훈련 데이터(X_train): {X_train.shape}")
print(f"  - 테스트 데이터(X_test): {X_test.shape}")

# 2. 모델 정의 (Random Forest)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=SEED,
        n_jobs=-1,          # (모든 CPU 사용)
        max_depth=10,       # (너무 깊지 않게, 과적합 방지)
        class_weight='balanced' # (y=0(70%)과 y=1(30%)의 불균형 처리)
    ))
])
print("  - 모델 파이프라인 정의 완료.")

In [ ]:
#  LGBM, CatBoost 추가 투입 및 비교 평가
print("\n--- [7/9] LGBM, CatBoost ---")
print("  - 기존 '랜포'와 비교하여 누가 더 뛰어난지 평가합니다.")

# 1. 추가 라이브러리 임포트 (아직 없으면 설치 후 커널 재시작)
# !pip install lightgbm catboost
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 2. 모델 정의 (RandomForest, LGBM, CatBoost)
models = {
    'RandomForest': RandomForestClassifier(random_state=SEED, n_jobs=-1, max_depth=10, class_weight='balanced'),
    'LightGBM': LGBMClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced'),
    'CatBoost': CatBoostClassifier(random_state=SEED, verbose=0, class_weights={0: 0.627, 1: 0.373}) # (불균형 데이터 비율)
}

results = {}

# 3. 각 모델 훈련 및 평가
for name, classifier in models.items():
    print(f"\n--- [{name}] 훈련 시작 ---")

    # 3-1. 파이프라인 구성 (전처리 + 모델)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor), # (이전 셀에서 정의한 preprocessor 재사용)
        ('classifier', classifier)
    ])

    # 3-2. 훈련
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    end_time = time.time()
    print(f"  - 훈련 완료! (소요 시간: {end_time - start_time:.2f} 초)")

    # 3-3. 예측 및 평가
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['0: 실행', '1: 선택'], output_dict=True)

    results[name] = {
        'accuracy': accuracy,
        'report': report,
        'pipeline': pipeline # (앙상블을 위해 파이프라인 저장)
    }

    print(f"\n🏆 [{name}] 예측 성능 평가:")
    print(classification_report(y_test, y_pred, target_names=['0: 실행', '1: 선택']))

# 4. 모델별 성능 요약
print("\n" + "="*50)
print("🏆 [모델별 최종 성능 요약]")
print("="*50)
for name, res in results.items():
    print(f"  - [{name}] 정확도(Accuracy): {res['accuracy']:.4f}")
    print(f"    - '선택(1)' F1-Score: {res['report']['1: 선택']['f1-score']:.4f}")
    print(f"    - '선택(1)' 정밀도(Precision): {res['report']['1: 선택']['precision']:.4f}")
    print(f"    - '선택(1)' 재현율(Recall): {res['report']['1: 선택']['recall']:.4f}")
print("="*50)

### 📊 [실행 결과] 'AI 수락률(KPI)' 예측 핵심 변수 (Feature Importance)
```text
==================================================
 LGBM 기반 'AI 수락률'에 영향을 미치는 핵심 변수 (Top 8)
==================================================
Feature_Group     Importance Score
--------------------------------------------------
field                   694
maintenance             453
initial                 383
position                317
llm                     215
is                      151
trigger                  79
--------------------------------------------------
> [시각화] 'LGBM'의 핵심 변수 중요도(Feature Importance) 분석 완료
> 'feature_importance_lgbm.png' 시각화 자료 저장 완료

### 💡 분석 결론

데이터 분석 결과, 사용자가 요구한 **적용 분야(`field`: 논문, 이메일, 마케팅 등)**와 **교정 강도(`maintenance`: 단순 맞춤법, 일부 고쳐 쓰기, 전체 다시 쓰기)**가 수락률(KPI) 및 이탈에 가장 큰 영향을 미치는 핵심 변수(Feature)로 확인되었습니다. 
*(※ 단, `tone` 변수의 경우 해당 스타트업의 서비스 정책 변경으로 기능이 삭제되어 향후 파이프라인에서는 제외함)*

**-> 즉, 복잡한 맥락 파악이 필요하고 교정 난이도가 높은 요청(논문, 전체 다시 쓰기 등)을 가벼운 모델(Gemini Flash)로 일괄 처리하여 품질이 떨어지는 현상이 고객 이탈의 핵심 원인입니다.**

**-> 해결책: 사용자의 요청 데이터(적용 분야, 교정 강도 등)를 기반으로 문장의 교정 난이도를 사전에 평가하여, '어려운 문장'만 고성능 대형 모델로 우회시키는 동적 라우팅 파이프라인(SmartRouter) 설계가 필요합니다.**
